In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement Monte Carlo integration on a GPU. Given a set of function values $y_i = f(x_i)$ sampled at random points $x_i$ uniformly distributed in the interval $[a, b]$, estimate the definite integral:
  $$ \int_a^b f(x) \, dx \approx (b - a) \cdot \frac{1}{n} \sum_{i=1}^{n} y_i $$

  The Monte Carlo method approximates the integral by computing the average of the function values and multiplying by the interval width.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>result</code> variable</li>
  <li>Solutions are tested with absolute tolerance of 1e-2 and relative tolerance of 1e-2</li>
</ul>

<h2>Example:</h2>
<pre>
Input:  a = 0, b = 2, n_samples = 8
        y_samples = [0.0625, 0.25, 0.5625, 1.0, 1.5625, 2.25, 3.0625, 4.0]
Output: result = 3.1875
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>n_samples</code> ≤ 100,000,000</li>
  <li>-1000.0 ≤ <code>a</code> &lt; <code>b</code> ≤ 1000.0</li>
  <li>-10000.0 ≤ function values ≤ 10000.0</li>
  <li>The tolerance is set to 1e-2 to account for the inherent randomness in Monte Carlo methods and floating-point precision variations.</li>

  <li>Performance is measured with <code>n_samples</code> = 10,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// y_samples, result are device pointers
extern "C" void solve(const float* y_samples, float* result, float a, float b, int n_samples) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# y_samples, result are tensors on the GPU
@cute.jit
def solve(
    y_samples: cute.Tensor,
    result: cute.Tensor,
    a: cute.Float32,
    b: cute.Float32,
    n_samples: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# y_samples is a tensor on the GPU
@jax.jit
def solve(y_samples: jax.Array, a: float, b: float, n_samples: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# y_samples, result are device pointers
@export
def solve(
    y_samples: UnsafePointer[Float32, MutExternalOrigin],
    result: UnsafePointer[Float32, MutExternalOrigin],
    a: Float32,
    b: Float32,
    n_samples: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# y_samples, result are tensors on the GPU
def solve(y_samples: torch.Tensor, result: torch.Tensor, a: float, b: float, n_samples: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# y_samples, result are tensors on the GPU
def solve(y_samples: torch.Tensor, result: torch.Tensor, a: float, b: float, n_samples: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Monte Carlo Integration", atol=1e-02, rtol=1e-02, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self, y_samples: torch.Tensor, result: torch.Tensor, a: float, b: float, n_samples: int
    ):
        assert y_samples.shape == (n_samples,)
        assert result.shape == (1,)
        assert y_samples.dtype == result.dtype
        assert y_samples.device == result.device
        assert b > a

        # Monte Carlo integration: integral ≈ (b - a) * mean(y_samples)
        mean_y = torch.mean(y_samples)
        integral = (b - a) * mean_y

        result[0] = integral

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "y_samples": (ctypes.POINTER(ctypes.c_float), "in"),
            "result": (ctypes.POINTER(ctypes.c_float), "out"),
            "a": (ctypes.c_float, "in"),
            "b": (ctypes.c_float, "in"),
            "n_samples": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        y_samples = torch.tensor(
            [0.0625, 0.25, 0.5625, 1.0, 1.5625, 2.25, 3.0625, 4.0], device="cuda", dtype=dtype
        )
        result = torch.zeros(1, device="cuda", dtype=dtype)
        return {
            "y_samples": y_samples,
            "result": result,
            "a": 0.0,
            "b": 2.0,
            "n_samples": 8,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        test_specs = [
            # Basic test cases
            ("basic_8", [0.0625, 0.25, 0.5625, 1.0, 1.5625, 2.25, 3.0625, 4.0], 0.0, 2.0),
            ("constant_function", [1.0, 1.0, 1.0, 1.0], 0.0, 4.0),
            ("linear_function", [0.0, 1.0, 2.0, 3.0], 0.0, 3.0),
            ("negative_interval", [-1.0, -2.0, -3.0], -2.0, 1.0),
            ("small_interval", [0.5, 1.5], 1.0, 2.0),
        ]

        test_cases = []
        for _, y_vals, a, b in test_specs:
            n_samples = len(y_vals)
            test_cases.append(
                {
                    "y_samples": torch.tensor(y_vals, device="cuda", dtype=dtype),
                    "result": torch.zeros(1, device="cuda", dtype=dtype),
                    "a": a,
                    "b": b,
                    "n_samples": n_samples,
                }
            )

        # Random test cases with different sizes
        for _, n_samples, a, b in [
            ("small_samples", 10, 0.0, 1.0),
            ("medium_samples", 100, -1.0, 1.0),
            ("large_samples", 1000, 0.0, 10.0),
            ("many_samples", 10000, -5.0, 5.0),
        ]:
            test_cases.append(
                {
                    "y_samples": torch.empty(n_samples, device="cuda", dtype=dtype).uniform_(
                        -10.0, 10.0
                    ),
                    "result": torch.zeros(1, device="cuda", dtype=dtype),
                    "a": a,
                    "b": b,
                    "n_samples": n_samples,
                }
            )

        # Edge cases
        for _, n_samples, a, b in [
            ("min_samples", 1, 0.0, 1.0),
            ("large_interval", 100, -100.0, 100.0),
            ("small_interval_edge", 50, 0.0, 0.1),
        ]:
            test_cases.append(
                {
                    "y_samples": torch.empty(n_samples, device="cuda", dtype=dtype).uniform_(
                        -1.0, 1.0
                    ),
                    "result": torch.zeros(1, device="cuda", dtype=dtype),
                    "a": a,
                    "b": b,
                    "n_samples": n_samples,
                }
            )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        n_samples = 10000000
        return {
            "y_samples": torch.empty(n_samples, device="cuda", dtype=dtype).uniform_(
                -1000.0, 1000.0
            ),
            "result": torch.zeros(1, device="cuda", dtype=dtype),
            "a": -10.0,
            "b": 10.0,
            "n_samples": n_samples,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
